# BP GPU Test Suite - XTTS, X-Voice, OmniVoice on T4
Tests voice cloning engines on free Kaggle T4 (16GB VRAM).
Run time: ~15-20 min per engine.
Python 3.10/3.11 compatible.

In [ ]:
!nvidia-smi
import sys
print(f"Python: {sys.version}")
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Install dependencies with Python version constraints
import sys
py_version = f"{sys.version_info.major}.{sys.version_info.minor}"
print(f"Installing for Python {py_version}")

# For Python 3.10/3.11, use compatible versions
!pip install -U pip setuptools wheel
!pip install 'coqui-tts>=0.17.0,<0.25.0' --verbose 2>&1 | tail -20
!pip install onnxruntime-gpu --verbose 2>&1 | tail -10
!pip install soundfile librosa numpy scipy openai-whisper --verbose 2>&1 | tail -10

In [ ]:
# Verify imports
try:
    from TTS.api import TTS
    print("Coqui TTS import OK")
    TTS_AVAILABLE = True
except Exception as e:
    print(f"Coqui TTS not available: {e}")
    TTS_AVAILABLE = False

import onnxruntime as ort
print(f"ONNX providers: {ort.get_available_providers()}")
import whisper
print("Whisper import OK")

In [ ]:
# Test 1: XTTS v2 (Coqui)
import time, torch, soundfile as sf, numpy as np

text = "Včera ráno som išiel na trh kúpiť čerstvý chlieb a mlieko. Stretol som tam starého priateľa Ľuba, ktorý predával med a syry."
speaker_wav = "https://github.com/coqui-ai/TTS/releases/download/v0.17.0/xtts_v2_reference.wav"

if TTS_AVAILABLE:
    print("Loading XTTS v2...")
    t0 = time.time()
    xtts = TTS("tts_models/multilingual/multi-dataset/xtts_v2", gpu=True)
    print(f"Load time: {time.time()-t0:.1f}s")
    
    print("Synthesizing (zero-shot cloning)...")
    t0 = time.time()
    wav = xtts.tts(text=text, speaker_wav=speaker_wav, language="sk")
    syn_time = time.time() - t0
    audio_s = len(wav) / 22050
    print(f"Syn: {syn_time:.2f}s, Audio: {audio_s:.2f}s, RTF: {syn_time/audio_s:.3f}")
    sf.write("/kaggle/working/xtts_output.wav", np.array(wav), 22050)
    print("XTTS DONE")
else:
    print("Skipping XTTS - TTS not available")

In [ ]:
# Test 2: OmniVoice
try:
    from omnivoice import OmniVoice
    print("Loading OmniVoice...")
    t0 = time.time()
    omni = OmniVoice.from_pretrained("k2-fsa/OmniVoice", device_map="cuda", dtype=torch.float16)
    print(f"Load time: {time.time()-t0:.1f}s")
    
    text = "Včera ráno som išiel na trh kúpiť čerstvý chlieb a mlieko. Stretol som tam starého priateľa Ľuba, ktorý predával med a syry."
    ref_audio = "https://github.com/k2-fsa/OmniVoice/releases/download/v0.1/ref_sk.wav"
    ref_text = "Včera ráno som išiel na trh kúpiť čerstvý chlieb a mlieko."
    
    print("Synthesizing (zero-shot)...")
    t0 = time.time()
    prompt = omni.create_voice_clone_prompt(ref_audio, ref_text)
    prep_time = time.time() - t0
    t0 = time.time()
    audio = omni.generate(text=text, language="sk", voice_clone_prompt=prompt, num_step=12)[0]
    syn_time = time.time() - t0
    audio_s = len(audio) / omni.sampling_rate
    print(f"Prep: {prep_time:.2f}s, Syn: {syn_time:.2f}s, Audio: {audio_s:.2f}s, RTF: {syn_time/audio_s:.3f}")
    sf.write("/kaggle/working/omnivoice_output.wav", audio.cpu().numpy(), omni.sampling_rate)
    print("OmniVoice DONE")
except Exception as e:
    print(f"OmniVoice skipped: {e}")

In [ ]:
# Test 3: X-Voice via onyx (if model available)
try:
    import onnxruntime as ort
    print(f"ONNX Runtime providers: {ort.get_available_providers()}")
    # onyx RVC would go here if .onyx model available
except Exception as e:
    print(f"ONNX test skipped: {e}")

In [ ]:
# STT quality check on synthesized outputs
import whisper
model = whisper.load_model("small", device="cuda")

for f in ["/kaggle/working/xtts_output.wav", "/kaggle/working/omnivoice_output.wav"]:
    try:
        result = model.transcribe(f, language="sk")
        print(f"{f}: {result['text'][:100]}...")
    except:
        pass

In [ ]:
# Summary
import json, os
print("=== GPU TEST COMPLETE ===")